# Session 8: Capstone Case Study — Telecom Domain Agent
### Agentic AI Nano Bootcamp | Day 2, Session 8

---

## Learning Objectives

By the end of this session, you will be able to:
- Evaluate a small or lightweight GenAI model on accuracy, latency, and cost dimensions
- Select an appropriate model for a production agentic application
- Build a complete end-to-end telecom task automation bot using LangGraph
- Compose all patterns learned across Sessions 1-7 into a single working system
- Present a working agent to peers with a structured walkthrough

## Session Outline

1. Capstone Brief and Architecture Overview
2. Part 1: Model Evaluation Framework
3. Part 2: System Design — Telecom Task Automation Bot
4. Part 3: Build — Ingestion and Knowledge Base
5. Part 4: Build — LangGraph Agent with Full Tool Suite
6. Part 5: Build — Customer-Facing Q&A Interface
7. Part 6: Integration Test
8. Team Demo and Debrief

---

## The Capstone Brief

**SwiftNet Telecom** (fictional) needs an AI-powered task automation bot to handle three core operations:

| Function | Description |
|---|---|
| Fault ticket classification | Automatically classify incoming fault tickets by type, severity, and team |
| Customer Q&A | Answer subscriber queries about plans, coverage, and outages using a knowledge base |
| Plan recommendation | Recommend the best plan based on usage patterns and customer profile |

The bot must be built using LangGraph and deployed as a single invokable agent.

---

In [ ]:
import subprocess, os, json, time, random, datetime, operator
subprocess.run(['pip', 'install', 'langgraph', 'langchain', 'langchain-openai',
                'langchain-community', 'chromadb', 'openai', 'python-dotenv',
                'pandas', 'matplotlib', 'tabulate', '-q'], capture_output=True)

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List, Optional

plain_client = OpenAI()
print("All dependencies ready.")

## Part 1: Model Evaluation Framework

Before building the production agent, we evaluate candidate models on the dimensions that matter for a telecom support application.

### Evaluation Dimensions

| Dimension | Why It Matters | Measurement |
|---|---|---|
| **Task accuracy** | Correctness on domain-specific classification and Q&A | Automated scoring against ground truth |
| **Latency** | Customer-facing bots must respond in under 3 seconds | Timed API calls |
| **Cost per 1K tokens** | Determines operational budget | OpenAI pricing |
| **Context handling** | Ability to reason over long support transcripts | Long context test |
| **Instruction following** | Reliably returns structured JSON | Format compliance rate |

### Models Under Evaluation

| Model | Pricing (input/output per 1M tokens) | Context | Tier |
|---|---|---|---|
| gpt-4o | $5 / $15 | 128K | Premium |
| gpt-4o-mini | $0.15 / $0.60 | 128K | Cost-efficient |
| gpt-3.5-turbo | $0.50 / $1.50 | 16K | Legacy |

For open-source alternatives (Mistral 7B, Llama 3), the same evaluation applies but models run via Hugging Face Inference API or locally.

In [ ]:
# Evaluation dataset: ticket classification ground truth

EVAL_TICKETS = [
    {"text": "Customer reports complete loss of internet for 6 hours. Router lights: power solid green, internet blinking red.",
     "expected_category": "connectivity", "expected_priority": "P1"},
    {"text": "Invoice shows a Rs 200 charge labelled 'Value Added Services'. Customer did not subscribe to any VAS.",
     "expected_category": "billing", "expected_priority": "P3"},
    {"text": "Hospital network gateway offline. Patient monitoring systems disconnected. ICU affected.",
     "expected_category": "connectivity", "expected_priority": "P1"},
    {"text": "Customer wants to upgrade from 50 Mbps to 200 Mbps plan. Requesting information on pricing.",
     "expected_category": "plan_change", "expected_priority": "P4"},
    {"text": "Intermittent WiFi drops between 8 PM and 11 PM daily for the past week. Speed tests normal during the day.",
     "expected_category": "connectivity", "expected_priority": "P2"},
    {"text": "Customer received a final disconnection notice but claims to have paid the bill 3 days ago.",
     "expected_category": "billing", "expected_priority": "P2"},
    {"text": "Request to update registered mobile number and email address on account TN-2984712.",
     "expected_category": "account", "expected_priority": "P4"},
    {"text": "New connection request for a 5-floor commercial office building in Chennai. 200+ users expected.",
     "expected_category": "new_connection", "expected_priority": "P3"},
]

CLASSIFICATION_SCHEMA = """
Classify this telecom support ticket. Return ONLY valid JSON:
{"category": "<connectivity|billing|plan_change|account|new_connection|hardware|other>",
 "priority": "<P1|P2|P3|P4>",
 "team": "<NOC-L1|NOC-L2|Field-Engineering|Billing-Ops|Sales|Account-Management>"}

Priority guide: P1=critical/safety, P2=severe/business impact, P3=moderate, P4=low/informational

Ticket: {ticket}
"""

print(f"Evaluation dataset: {len(EVAL_TICKETS)} labelled tickets")

In [ ]:
# Model evaluation runner

import time, json

def evaluate_model(model_name: str, tickets: list) -> dict:
    """
    Evaluate a model on ticket classification.
    Returns accuracy, latency, and format compliance metrics.
    """
    results = []
    latencies = []

    for ticket in tickets:
        prompt = CLASSIFICATION_SCHEMA.format(ticket=ticket["text"])
        start  = time.time()
        try:
            response = plain_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=120,
                response_format={"type": "json_object"}
            )
            latency_ms = (time.time() - start) * 1000
            raw        = response.choices[0].message.content
            parsed     = json.loads(raw)
            category_correct  = parsed.get("category")  == ticket["expected_category"]
            priority_correct  = parsed.get("priority")  == ticket["expected_priority"]
            format_valid      = all(k in parsed for k in ["category", "priority", "team"])
            results.append({
                "category_correct": category_correct,
                "priority_correct": priority_correct,
                "format_valid":     format_valid,
                "latency_ms":       latency_ms,
                "tokens":           response.usage.total_tokens
            })
            latencies.append(latency_ms)
        except Exception as e:
            results.append({"category_correct": False, "priority_correct": False,
                            "format_valid": False, "latency_ms": 5000, "tokens": 0})

    n = len(results)
    return {
        "model":             model_name,
        "category_accuracy": round(sum(r["category_correct"] for r in results) / n * 100, 1),
        "priority_accuracy": round(sum(r["priority_correct"] for r in results) / n * 100, 1),
        "format_compliance": round(sum(r["format_valid"]     for r in results) / n * 100, 1),
        "avg_latency_ms":    round(sum(r["latency_ms"]       for r in results) / n, 0),
        "p95_latency_ms":    round(sorted(latencies)[int(n * 0.95)] if latencies else 0, 0),
        "avg_tokens":        round(sum(r["tokens"]           for r in results) / n, 0),
    }

# Evaluate available models
models_to_eval = ["gpt-4o-mini", "gpt-4o"]

eval_results = []
for model in models_to_eval:
    print(f"Evaluating {model}...")
    result = evaluate_model(model, EVAL_TICKETS)
    eval_results.append(result)
    print(f"  Category accuracy: {result['category_accuracy']}%")
    print(f"  Priority accuracy: {result['priority_accuracy']}%")
    print(f"  Avg latency:       {result['avg_latency_ms']} ms")

print("\nEvaluation complete.")

In [ ]:
# Visualise evaluation results
import matplotlib.pyplot as plt
import numpy as np

if eval_results:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('Model Evaluation Results', fontsize=13, fontweight='bold')

    model_names = [r["model"] for r in eval_results]
    colors = ['#1F4E79', '#1B5E3B', '#7B3F00', '#6C3483']

    # Accuracy
    ax = axes[0]
    x  = np.arange(len(model_names))
    w  = 0.35
    ax.bar(x - w/2, [r["category_accuracy"] for r in eval_results], w,
           label='Category', color=colors[0], alpha=0.85)
    ax.bar(x + w/2, [r["priority_accuracy"]  for r in eval_results], w,
           label='Priority',  color=colors[1], alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, fontsize=9)
    ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Classification Accuracy'); ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    for xi, y in zip(x - w/2, [r["category_accuracy"] for r in eval_results]):
        ax.text(xi, y + 1, f'{y}%', ha='center', fontsize=8)
    for xi, y in zip(x + w/2, [r["priority_accuracy"] for r in eval_results]):
        ax.text(xi, y + 1, f'{y}%', ha='center', fontsize=8)

    # Latency
    ax = axes[1]
    ax.bar(model_names, [r["avg_latency_ms"] for r in eval_results],
           color=colors[2], alpha=0.85)
    ax.set_ylabel('Avg latency (ms)')
    ax.set_title('Response Latency')
    ax.axhline(3000, color='red', linestyle='--', linewidth=1, label='3s threshold')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    for i, r in enumerate(eval_results):
        ax.text(i, r["avg_latency_ms"] + 50, f'{int(r["avg_latency_ms"])}ms',
                ha='center', fontsize=8)

    # Format compliance
    ax = axes[2]
    ax.bar(model_names, [r["format_compliance"] for r in eval_results],
           color=colors[3], alpha=0.85)
    ax.set_ylabel('Format compliance (%)')
    ax.set_title('Instruction Following')
    ax.set_ylim(0, 110)
    ax.spines[['top','right']].set_visible(False)
    for i, r in enumerate(eval_results):
        ax.text(i, r["format_compliance"] + 1, f'{r["format_compliance"]}%',
                ha='center', fontsize=8)

    plt.tight_layout()
    plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print recommendation
    best = max(eval_results, key=lambda r: r["category_accuracy"] + r["priority_accuracy"] - r["avg_latency_ms"]/100)
    print(f"\nRecommended model: {best['model']}")
    print(f"Rationale: highest combined accuracy with acceptable latency for customer-facing use.")

## Part 2: System Design — Telecom Task Automation Bot

### Architecture

```
User input
     |
     v
 [intent_router]          Classifies intent: ticket_classification | customer_qa | plan_recommendation
     |
     +-------> [ticket_classification_node]  -> [create_ticket_node]  -> [respond_node] -> END
     |
     +-------> [rag_retrieve_node]           -> [rag_generate_node]   -> [respond_node] -> END
     |
     +-------> [profile_analysis_node]       -> [recommend_node]      -> [respond_node] -> END
```

### Component Responsibilities

| Component | Responsibility | Technology |
|---|---|---|
| Intent router | Determine which workflow to invoke | LLM + conditional edges |
| Ticket classification | Classify, prioritise, and assign tickets | LLM + structured output |
| Create ticket | Write classified ticket to the system | Python function (simulated) |
| RAG retrieve | Find relevant knowledge base documents | ChromaDB + embeddings |
| RAG generate | Produce grounded answer from context | LLM + retrieved docs |
| Profile analysis | Analyse usage data for plan matching | LLM + customer data |
| Plan recommend | Select and justify the best plan | LLM + plan catalogue |
| Respond | Format the final response for the customer | LLM + formatting |

### Data Models

The shared agent state carries all information across nodes, eliminating the need for explicit argument passing.

In [ ]:
# Shared state and tool definitions for the capstone bot

class TelecomBotState(TypedDict):
    # Input
    user_input:         str
    customer_id:        str
    # Routing
    intent:             str          # ticket_classification | customer_qa | plan_recommendation
    # Ticket classification outputs
    ticket_category:    str
    ticket_priority:    str
    ticket_team:        str
    ticket_id:          str
    # RAG outputs
    retrieved_docs:     List[str]
    rag_answer:         str
    # Plan recommendation outputs
    customer_profile:   dict
    recommended_plan:   str
    recommendation_reason: str
    # Final
    final_response:     str
    messages:           Annotated[list, operator.add]


# ── Simulated backend functions (tools) ──

PLAN_CATALOGUE = [
    {"name": "Basic 50",    "speed_mbps": 50,   "price_inr": 399,  "data_gb": "Unlimited", "best_for": "Light users, 1-2 devices"},
    {"name": "Standard 100","speed_mbps": 100,  "price_inr": 699,  "data_gb": "Unlimited", "best_for": "Families, 3-4 devices, HD streaming"},
    {"name": "Pro 200",      "speed_mbps": 200,  "price_inr": 999,  "data_gb": "Unlimited", "best_for": "Power users, WFH, 4K streaming"},
    {"name": "Ultra 500",   "speed_mbps": 500,  "price_inr": 1499, "data_gb": "Unlimited", "best_for": "Gamers, large families, smart home"},
    {"name": "Business 1G", "speed_mbps": 1000, "price_inr": 3499, "data_gb": "Unlimited", "best_for": "SME, office use, guaranteed SLA"},
]

def get_customer_profile(customer_id: str) -> dict:
    """Retrieve customer usage profile (simulated)."""
    random.seed(hash(customer_id) % 997)
    return {
        "customer_id":       customer_id,
        "current_plan":      random.choice(["Basic 50", "Standard 100", "Pro 200"]),
        "avg_monthly_usage_gb": random.randint(150, 900),
        "peak_speed_used_mbps": random.randint(20, 180),
        "device_count":      random.randint(2, 8),
        "wfh_flag":          random.choice([True, False]),
        "streaming_flag":    random.choice([True, True, False]),
        "complaints_6m":     random.randint(0, 3),
    }

def create_ticket(category: str, priority: str, team: str,
                  description: str, customer_id: str) -> dict:
    """Create a ticket in the NOC system (simulated)."""
    ticket_id = f"TKT-{random.randint(100000, 999999)}"
    return {
        "ticket_id":  ticket_id,
        "status":     "created",
        "category":   category,
        "priority":   priority,
        "team":       team,
        "created_at": datetime.datetime.now().isoformat()
    }

print(f"State schema defined. Plan catalogue: {len(PLAN_CATALOGUE)} plans.")
print("Sample customer profile:")
import pprint
pprint.pprint(get_customer_profile("TN-2984712"))

## Part 3: Build — Knowledge Base for Q&A

In [ ]:
# Build the customer-facing knowledge base

KNOWLEDGE_BASE = [
    "SwiftNet plans range from Basic 50 Mbps at Rs 399/month to Business 1 Gbps at Rs 3499/month. All plans include unlimited data.",
    "To report an outage, call 1800-XXX-XXXX or submit a ticket via the SwiftNet app. P1 outages are responded to within 2 hours.",
    "Plan upgrades are effective from the next billing cycle. Downgrades require 30 days notice per terms of service.",
    "SwiftNet 5G coverage is available in Chennai, Mumbai, Bangalore, Hyderabad, and Delhi as of 2025. Full coverage maps are on swiftnet.in/coverage.",
    "The SwiftNet router uses WPA3 security. Default WiFi password is printed on the back of the device. To change it, log in to 192.168.1.1.",
    "Billing cycles run on the 1st of each month. Bills are sent via email and are viewable in the SwiftNet app under Account > Bills.",
    "Data usage can be monitored in real time in the SwiftNet app. Usage resets at midnight on the 1st of each month.",
    "SwiftNet provides a 7-day SLA for new connection installation from the date of payment confirmation.",
    "For business customers, a dedicated account manager is assigned and available on a direct line 9 AM to 7 PM on working days.",
    "To cancel service, submit a written request via email to cancel@swiftnet.in or visit a SwiftNet service centre with valid ID.",
    "SwiftNet's fiber network uses GPON technology for speeds up to 1 Gbps symmetric with typical latency under 5ms.",
    "Monthly speed test results below 80% of subscribed speed qualify for a service credit under SwiftNet's SLA guarantee.",
]

embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(texts=KNOWLEDGE_BASE, embedding=embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Knowledge base indexed: {len(KNOWLEDGE_BASE)} documents")

# Test retrieval
test_q   = "How do I change my WiFi password?"
test_docs = retriever.invoke(test_q)
print(f"\nTest retrieval for '{test_q}':")
for d in test_docs:
    print(f"  - {d.page_content[:80]}...")

## Part 4: Build — LangGraph Agent

In [ ]:
llm      = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
llm_fast = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# ── Node 1: Intent Router ──
def intent_router_node(state: TelecomBotState) -> dict:
    prompt = f"""
Classify the following customer input into exactly one intent.
Return only the intent word — nothing else.

Intents:
  ticket_classification   — reporting a fault, outage, or technical problem
  customer_qa             — asking a question about plans, features, billing, or policy
  plan_recommendation     — asking for a plan recommendation or wanting to upgrade/change plan

Input: {state['user_input']}
"""
    response = llm_fast.invoke([HumanMessage(content=prompt)])
    intent   = response.content.strip().lower()
    if intent not in ("ticket_classification", "customer_qa", "plan_recommendation"):
        intent = "customer_qa"
    print(f"  [intent_router] Intent: {intent}")
    return {"intent": intent}

# ── Node 2: Ticket Classification ──
def ticket_classification_node(state: TelecomBotState) -> dict:
    prompt = f"""
Classify this telecom fault report. Return only valid JSON.
{{"category": "<connectivity|billing|hardware|account|other>",
  "priority": "<P1|P2|P3|P4>",
  "team": "<NOC-L1|NOC-L2|Field-Engineering|Billing-Ops|Account-Management>",
  "summary": "<one sentence summary>"}}

Priority: P1=critical/safety, P2=business impact, P3=moderate, P4=low
Fault report: {state['user_input']}
"""
    raw = llm_fast.invoke(
        [HumanMessage(content=prompt)]
    ).content
    try:
        parsed = json.loads(raw.strip().strip('`').replace('json','').strip())
    except Exception:
        parsed = {"category": "other", "priority": "P3", "team": "NOC-L1", "summary": raw[:100]}
    print(f"  [ticket_classification] {parsed.get('category')} / {parsed.get('priority')}")
    return {
        "ticket_category": parsed.get("category", "other"),
        "ticket_priority": parsed.get("priority", "P3"),
        "ticket_team":     parsed.get("team", "NOC-L1"),
    }

# ── Node 3: Create Ticket ──
def create_ticket_node(state: TelecomBotState) -> dict:
    result = create_ticket(
        category    = state["ticket_category"],
        priority    = state["ticket_priority"],
        team        = state["ticket_team"],
        description = state["user_input"],
        customer_id = state.get("customer_id", "UNKNOWN")
    )
    print(f"  [create_ticket] Ticket created: {result['ticket_id']}")
    return {"ticket_id": result["ticket_id"]}

# ── Node 4: RAG Retrieve ──
def rag_retrieve_node(state: TelecomBotState) -> dict:
    docs = retriever.invoke(state["user_input"])
    doc_texts = [d.page_content for d in docs]
    print(f"  [rag_retrieve] Retrieved {len(doc_texts)} documents")
    return {"retrieved_docs": doc_texts}

# ── Node 5: RAG Generate ──
def rag_generate_node(state: TelecomBotState) -> dict:
    context = "\n".join(f"- {d}" for d in state["retrieved_docs"])
    prompt = f"""
You are SwiftNet's AI support assistant. Answer the customer question using only the provided context.
If the context does not contain the answer, say you will connect them with a human agent.
Be concise, friendly, and specific. Maximum 100 words.

Context:
{context}

Customer question: {state['user_input']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [rag_generate] Answer generated")
    return {"rag_answer": response.content}

# ── Node 6: Profile Analysis ──
def profile_analysis_node(state: TelecomBotState) -> dict:
    profile = get_customer_profile(state.get("customer_id", "CX-DEFAULT"))
    print(f"  [profile_analysis] Profile loaded: {profile['current_plan']} plan, {profile['device_count']} devices")
    return {"customer_profile": profile}

# ── Node 7: Plan Recommendation ──
def plan_recommendation_node(state: TelecomBotState) -> dict:
    profile  = state.get("customer_profile", {})
    catalogue = json.dumps(PLAN_CATALOGUE, indent=2)
    prompt = f"""
You are a SwiftNet plan advisor. Based on the customer profile, select the single best plan from the catalogue.
Consider: current usage, device count, WFH status, streaming needs, and budget sensitivity.

Customer profile:
{json.dumps(profile, indent=2)}

Plan catalogue:
{catalogue}

Return ONLY JSON: {{"recommended_plan": "<plan name>", "reason": "<two sentences justifying the choice>"}}
"""
    raw = llm_fast.invoke([HumanMessage(content=prompt)]).content
    try:
        parsed = json.loads(raw.strip().strip('`').replace('json','').strip())
    except Exception:
        parsed = {"recommended_plan": "Standard 100", "reason": raw[:200]}
    print(f"  [plan_recommendation] Recommended: {parsed.get('recommended_plan')}")
    return {
        "recommended_plan":        parsed.get("recommended_plan", ""),
        "recommendation_reason":   parsed.get("reason", ""),
    }

# ── Node 8: Respond ──
def respond_node(state: TelecomBotState) -> dict:
    intent = state.get("intent", "customer_qa")

    if intent == "ticket_classification":
        response = (
            f"Your fault report has been logged and assigned to our {state['ticket_team']} team. "
            f"Ticket reference: {state['ticket_id']} (Priority: {state['ticket_priority']}). "
            f"You will receive an update within "
            f"{'2 hours' if state['ticket_priority'] == 'P1' else '4-8 hours' if state['ticket_priority'] == 'P2' else '24 hours'}."
        )
    elif intent == "customer_qa":
        response = state.get("rag_answer", "I was unable to find an answer. Connecting you to a human agent.")
    elif intent == "plan_recommendation":
        plan   = state.get("recommended_plan", "")
        reason = state.get("recommendation_reason", "")
        plan_details = next((p for p in PLAN_CATALOGUE if p["name"] == plan), {})
        price = plan_details.get("price_inr", "")
        speed = plan_details.get("speed_mbps", "")
        response = (
            f"Based on your usage profile, I recommend the {plan} plan at Rs {price}/month ({speed} Mbps, unlimited data). "
            f"{reason}"
        )
    else:
        response = "Thank you for contacting SwiftNet. A support agent will be in touch shortly."

    print(f"  [respond] Response composed for intent: {intent}")
    return {"final_response": response}

print("All nodes defined.")

In [ ]:
# Build the complete telecom bot graph

def route_by_intent(state: TelecomBotState) -> str:
    intent_map = {
        "ticket_classification": "ticket_classification",
        "customer_qa":           "rag_retrieve",
        "plan_recommendation":   "profile_analysis",
    }
    return intent_map.get(state.get("intent", ""), "rag_retrieve")

builder = StateGraph(TelecomBotState)

# Register nodes
builder.add_node("intent_router",         intent_router_node)
builder.add_node("ticket_classification", ticket_classification_node)
builder.add_node("create_ticket",         create_ticket_node)
builder.add_node("rag_retrieve",          rag_retrieve_node)
builder.add_node("rag_generate",          rag_generate_node)
builder.add_node("profile_analysis",      profile_analysis_node)
builder.add_node("plan_recommendation",   plan_recommendation_node)
builder.add_node("respond",               respond_node)

# Entry point
builder.set_entry_point("intent_router")

# Routing from intent
builder.add_conditional_edges(
    "intent_router",
    route_by_intent,
    {
        "ticket_classification": "ticket_classification",
        "rag_retrieve":          "rag_retrieve",
        "profile_analysis":      "profile_analysis",
    }
)

# Ticket workflow
builder.add_edge("ticket_classification", "create_ticket")
builder.add_edge("create_ticket",         "respond")

# Q&A workflow
builder.add_edge("rag_retrieve", "rag_generate")
builder.add_edge("rag_generate", "respond")

# Plan recommendation workflow
builder.add_edge("profile_analysis",    "plan_recommendation")
builder.add_edge("plan_recommendation", "respond")

# All paths end at respond
builder.add_edge("respond", END)

telecom_bot = builder.compile()

print("Telecom bot graph compiled.")
print("Nodes:", list(telecom_bot.get_graph().nodes.keys()))

## Part 5 and 6: Integration Test

In [ ]:
# Integration test: run all three workflows end-to-end

DEFAULT_STATE = {
    "customer_id":          "TN-2984712",
    "user_input":           "",
    "intent":               "",
    "ticket_category":      "",
    "ticket_priority":      "",
    "ticket_team":          "",
    "ticket_id":            "",
    "retrieved_docs":       [],
    "rag_answer":           "",
    "customer_profile":     {},
    "recommended_plan":     "",
    "recommendation_reason": "",
    "final_response":       "",
    "messages":             []
}

test_inputs = [
    # Ticket classification
    "My internet has been completely down since 6 AM. I work from home and have missed two client calls.",
    # Customer Q&A
    "What is the installation time for a new connection?",
    # Plan recommendation
    "I have 6 devices at home, stream 4K on two TVs, and work from home full time. What plan should I get?",
]

for user_input in test_inputs:
    print("\n" + "=" * 65)
    print(f"Customer: {user_input}")
    print("=" * 65)

    state = {**DEFAULT_STATE, "user_input": user_input}
    result = telecom_bot.invoke(state)

    print(f"\n  Intent detected: {result['intent']}")
    print(f"\n  Bot response:")
    print(f"  {result['final_response']}")

In [ ]:
# Stress test: run a batch of queries and measure performance

import time, pandas as pd

BATCH_QUERIES = [
    ("What are your 5G coverage cities?",                               "customer_qa"),
    ("Router keeps disconnecting every 2 hours since firmware update.", "ticket_classification"),
    ("How do I download my bill?",                                      "customer_qa"),
    ("I want to switch to a faster plan.",                              "plan_recommendation"),
    ("Complete outage. No service at all. Urgent!",                     "ticket_classification"),
    ("What is the difference between Basic 50 and Standard 100?",       "customer_qa"),
]

batch_results = []
for query, expected_intent in BATCH_QUERIES:
    start  = time.time()
    result = telecom_bot.invoke({**DEFAULT_STATE, "user_input": query})
    elapsed = round((time.time() - start) * 1000)

    intent_correct = result["intent"] == expected_intent
    batch_results.append({
        "query":          query[:45],
        "expected":       expected_intent,
        "predicted":      result["intent"],
        "correct":        intent_correct,
        "latency_ms":     elapsed,
    })

df = pd.DataFrame(batch_results)
accuracy = df["correct"].mean() * 100

print("\nBatch Test Results")
print("=" * 80)
print(df.to_string(index=False))
print(f"\nIntent accuracy:    {accuracy:.1f}%")
print(f"Avg latency:        {df['latency_ms'].mean():.0f} ms")
print(f"Max latency:        {df['latency_ms'].max()} ms")

## Team Demo Guide

Each team will have **5 minutes** to demonstrate their agent. Use this structure:

### Demo Script (5 minutes)

**Minute 1 — Architecture walkthrough**
- Show the LangGraph workflow diagram
- Explain each node and the routing logic
- State which model was selected from the evaluation and why

**Minutes 2-4 — Live demonstration**
- Run one input from each of the three intents (ticket, Q&A, plan recommendation)
- Walk through the node execution trace in the output
- Show the final response for each

**Minute 5 — What would you build next?**
- Name one production improvement you would make
- Name one additional tool or workflow you would add

### Evaluation Criteria for Peer Review

| Criterion | Weight | What to Look For |
|---|---|---|
| Routing accuracy | 30% | Does intent detection work reliably? |
| Response quality | 30% | Are answers accurate, specific, and well-formatted? |
| Code clarity | 20% | Is the graph structure and node code readable? |
| Presentation | 20% | Is the demo walkthrough clear and confident? |

In [ ]:
# Your demo cell — run your own test queries here

YOUR_QUERY = "I have been a customer for 5 years and I am very unhappy with the repeated outages. I want to cancel."

print("Running your query through the telecom bot...\n")

result = telecom_bot.invoke({**DEFAULT_STATE, "user_input": YOUR_QUERY})

print("\n" + "=" * 65)
print(f"Input:    {YOUR_QUERY}")
print(f"Intent:   {result['intent']}")
print(f"\nResponse: {result['final_response']}")
print("=" * 65)

## Session Summary and Bootcamp Wrap-Up

### What Was Built in This Session

| Component | Technology | Pattern |
|---|---|---|
| Model evaluation | OpenAI API + timing | Systematic benchmarking |
| Intent routing | LangGraph conditional edges | Classifier-then-route |
| Ticket classification | Structured LLM output | Instructional prompting |
| Customer Q&A | ChromaDB + RAG | Retrieve-augment-generate |
| Plan recommendation | Profile analysis + LLM | Data-grounded generation |
| End-to-end composition | LangGraph StateGraph | Orchestrator pattern |

### What Was Covered Across the Full Bootcamp

| Session | Core Concept | Output |
|---|---|---|
| 1 | Generative AI fundamentals | Platform comparison, domain use case map |
| 2 | LLMs and APIs | Summarisation agent, translation agent |
| 3 | Prompt engineering | Optimised chatbot, prompt template library |
| 4 | Vector DB and RAG | Olympics Q&A system with LangChain |
| 5 | AI agents | Simple agent, reflective agent |
| 6 | Function calling and ReAct | NOC analyst agent with memory |
| 7 | LangGraph | Support routing graph, RAG-with-critique graph |
| 8 | Capstone | End-to-end telecom task automation bot |

### Recommended Next Steps

1. **Persist the vector store** to disk using `Chroma(persist_directory=...)` so the knowledge base survives restarts.
2. **Add LangGraph checkpointing** using `SqliteSaver` to enable conversation memory across sessions.
3. **Evaluate with larger datasets** — 8 tickets is a proof of concept; production evaluation needs 500+.
4. **Add human-in-the-loop** — LangGraph supports interrupt nodes that pause for human approval before critical actions.
5. **Deploy** — wrap the graph in a FastAPI endpoint and containerise with Docker for production use.

---
*Agentic AI Nano Bootcamp | Day 2, Session 8 — Capstone*